In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import duckdb
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED = DATA_DIR / 'processed'
MAP_DIR = PROJECT_ROOT / 'reports' / 'maps'
EDA_DIR = PROJECT_ROOT / 'reports'
MAP_DIR.mkdir(parents=True, exist_ok=True)
Listing_Data_Path = str(DATA_DIR / 'listings')
Review_Data_Path = str(DATA_DIR / 'reviews')
Output_Data_Path = str(PROCESSED)
PROCESSED.mkdir(parents=True, exist_ok=True)

con = duckdb.connect(str(PROJECT_ROOT / 'airbnb.duckdb'))
con.execute("SET memory_limit = '12GB'")  # suitable initial limit for a 32 GB machine
con.execute("SET threads = 4")
con.execute(f"SET temp_directory = '{str(PROJECT_ROOT / 'duckdb_temp').replace(chr(39), chr(39)*2)}'")

listings_path = PROCESSED / 'listings_master.parquet'
reviews_path = PROCESSED / 'reviews_clean.parquet'
if not listings_path.is_file():
    raise FileNotFoundError(f'Missing {listings_path}. Copy your listings_master.parquet here, or run optional integration script.')
if not reviews_path.is_file():
    raise FileNotFoundError(f'Missing {reviews_path}. Copy your reviews_clean.parquet here.')
listings_master = pd.read_parquet(listings_path)
for table_name, file_path in [('reviews_clean', reviews_path)]:
    quoted = str(file_path).replace("'", "''")
    con.execute(f"CREATE OR REPLACE VIEW {table_name} AS SELECT * FROM read_parquet('{quoted}')")
print(f'Listings: {len(listings_master):,}; reviews: queryable via DuckDB view reviews_clean')


In [ ]:
#average review_scores_rating
highest_average_review_score = (
    listings_master
    .groupby("location")["review_scores_rating"]
    .median()
    .nlargest(10)
    .sort_values()
)

print("median highest reviews")
display(highest_average_review_score)

lowest_average_review_score = (
    listings_master
    .groupby("location")["review_scores_rating"]
    .median()
    .nsmallest(10)
)

print("median lowest reviews")
display(lowest_average_review_score)


In [ ]:
#Review frequency per location by season
reviews_by_location_season = con.execute("""
    WITH seasonal_reviews AS (
        SELECT
            country,
            location,
            CASE
                WHEN MONTH(date) IN (12, 1, 2) THEN 'Winter'
                WHEN MONTH(date) IN (3, 4, 5) THEN 'Spring'
                WHEN MONTH(date) IN (6, 7, 8) THEN 'Summer'
                WHEN MONTH(date) IN (9, 10, 11) THEN 'Autumn'
            END AS season,

            CASE
                WHEN MONTH(date) = 12 THEN YEAR(date) + 1
                ELSE YEAR(date)
            END AS season_year

        FROM reviews_clean
        WHERE date IS NOT NULL
    )

    SELECT
        country,
        location,
        season_year,
        season,
        COUNT(*) AS total_reviews

    FROM seasonal_reviews

    GROUP BY
        country,
        location,
        season_year,
        season

    ORDER BY
        country,
        location,
        season_year,
        season
""").df()

display(reviews_by_location_season.head(20))

In [ ]:
#Seasonal review count
reviews_by_season = con.execute("""
    WITH seasonal_reviews AS (
        SELECT
            date,
            CASE
                WHEN MONTH(date) IN (12, 1, 2) THEN 'Winter'
                WHEN MONTH(date) IN (3, 4, 5) THEN 'Spring'
                WHEN MONTH(date) IN (6, 7, 8) THEN 'Summer'
                WHEN MONTH(date) IN (9, 10, 11) THEN 'Autumn'
            END AS season,

            CASE
                WHEN MONTH(date) = 12 THEN YEAR(date) + 1
                ELSE YEAR(date)
            END AS season_year

        FROM reviews_clean
        WHERE date IS NOT NULL
    )

    SELECT
        season_year,
        season,
        COUNT(*) AS total_reviews

    FROM seasonal_reviews

    GROUP BY season_year, season

    ORDER BY
        season_year,
        CASE season
            WHEN 'Winter' THEN 1
            WHEN 'Spring' THEN 2
            WHEN 'Summer' THEN 3
            WHEN 'Autumn' THEN 4
        END
""").df()

display(reviews_by_season)


In [ ]:
#Percentage of reviews per season
seasonal_share = reviews_by_location_season.copy()

seasonal_share["annual_reviews"] = (
    seasonal_share.groupby(
        ["location", "season_year"]
    )["total_reviews"].transform("sum")
)

seasonal_share["season_pct"] = (
    seasonal_share["total_reviews"]
    / seasonal_share["annual_reviews"]
    * 100
).round(2)

display(seasonal_share.head(20))

In [ ]:
reviews_location_summary = con.execute("""
    SELECT
        country,
        location,
        COUNT(*) AS total_reviews,
        COUNT(DISTINCT listing_id) AS reviewed_listings,
        MIN(date) AS earliest_review,
        MAX(date) AS latest_review
    FROM reviews_clean
    GROUP BY country, location
    ORDER BY total_reviews DESC
""").df()

display(reviews_location_summary)

In [ ]:
#Comparison of popular locations

locations = ["Barcelona", "London", "Paris", "Rome"]

# Select the most recent complete calendar year
analysis_year = 2025

plot_data = reviews_by_location_season[
    (reviews_by_location_season["location"].isin(locations)) &
    (reviews_by_location_season["season_year"] == analysis_year)
]

pivot_data = plot_data.pivot(
    index="season",
    columns="location",
    values="total_reviews"
)

pivot_data.plot(
    kind="bar",
    figsize=(12, 6)
)

plt.title(f"Seasonal Airbnb Review Activity by Location ({analysis_year})")
plt.xlabel("Season")
plt.ylabel("Number of Reviews")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

2. Host Data

In [ ]:
#Number of hosts and properties by location
host_summary = (
    listings_master
    .groupby("location")
    .agg(
        total_hosts=("host_id", "nunique"),
        total_properties=("id", "nunique")
    )
    .sort_values("total_hosts", ascending=False)
)

display (host_summary)

In [ ]:
#Median properties per host by country
host_property_count = (
    listings_master
    .groupby("country")["calculated_host_listings_count"]
    .median()
    .sort_values()
)

print("Median Properties per Host by Country")
display(host_property_count)

In [ ]:
#host activity by location
host_activity = (
    listings_master
    .groupby("country")["host_is_superhost"]
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .mul(100)
    .round(2)
)

host_activity.plot(kind="bar", stacked=True)

plt.title("Host Activity by Location")
plt.xlabel("Location")
plt.ylabel("Percentage")

3. Location Data

In [ ]:
#Plot of top locations by listing count
import matplotlib.pyplot as plt

location_counts = (
    listings_master["location"]
    .value_counts()
    .head(20)
    .sort_values()
)

plt.figure(figsize=(12, 8))
location_counts.plot(kind="barh")

plt.title("Top 20 European Airbnb Locations by Listing Count")
plt.xlabel("Number of Listings")
plt.ylabel("Location")
plt.tight_layout()
plt.show()

In [ ]:
#display of room types
display(
    listings_master["room_type"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

In [ ]:
#Chart of most common room type by location
room_type_counts = (
    listings_master
    .groupby("location")["room_type"]
    .value_counts()
    .head(20)
    .sort_values()
    .unstack(fill_value=0)
)

room_type_counts.plot(kind="bar", stacked=True)

plt.title("Most Common Room Types by Location")
plt.xlabel("Location")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.legend(title="Room Type")
plt.tight_layout()
plt.show()

4. Price Data

In [ ]:
#price summary
price_summary = (
    listings_master["price"]
    .describe()
    .to_frame()
    .T
    .round(2)
)

display(price_summary)

#check all have $ prefix
assert all(listings_master["price"].str.startswith("$"))
print("All prices have $ prefix.")

In [ ]:
#median prices in highest priced cities
median_prices_high = (
    listings_master
    .groupby("location")["price_quote_price_per_night"]
    .median()
    .nlargest(20)
    .sort_values()
)

display(median_prices_high)

#Bar chart for high-priced cities
median_prices_high.plot(kind="barh")

plt.title("Median Price per High-Price Cities")
plt.xlabel("Median Price")
plt.ylabel("City")
plt.tight_layout()
plt.show()

#Median prices per lowest price city
median_prices_low = (
    listings_master
    .groupby("location")["price_quote_price_per_night"]
    .median()
    .nsmallest(20)
    .sort_values()
)

display(median_prices_low)

median_prices_low.plot(kind="barh")

plt.title("Median Price per Low-Price Cities")
plt.xlabel("Median Price")
plt.ylabel("City")
plt.tight_layout()
plt.show()

In [ ]:
#yearly availability by country
availability_by_country = (
    listings_master
    .groupby("country")["availability_365"]
    .median()
    .sort_values()
)

print("Median Yearly Availability by Country")
display(availability_by_country)



In [ ]:
#graph of median price per country
median_price_by_country = (
    listings_master
    .groupby("country")["price_quote_price_per_night"]
    .median()
    .sort_values()
)

median_price_by_country.plot(kind="barh")

plt.title("Median Price per Country")
plt.xlabel("Median Price")
plt.ylabel("Country")
plt.tight_layout()
plt.show()

5. Geographic Data

In [ ]:
#Geographic data
geo_summary = listings_master[
    ["latitude", "longitude", "location",
     "country", "neighbourhood_cleansed"]
].copy()

display(geo_summary.describe(include="all"))
display(geo_summary.isna().sum())

In [ ]:
#Check coordinate valididty

import pandas as pd
import numpy as np

listings_master["latitude"] = pd.to_numeric(
    listings_master["latitude"], errors="coerce"
)

listings_master["longitude"] = pd.to_numeric(
    listings_master["longitude"], errors="coerce"
)

geo_data = listings_master.dropna(
    subset=["latitude", "longitude"]
).copy()

geo_data = geo_data[
    geo_data["latitude"].between(-90, 90)
    & geo_data["longitude"].between(-180, 180)
    & ~(
        (geo_data["latitude"] == 0)
        & (geo_data["longitude"] == 0)
    )
]

print("Listings with valid coordinates:", len(geo_data))

In [ ]:
#Competition by neighbourhood
neighbourhood_competition = (
    geo_data.groupby(
        ["country", "location", "neighbourhood_cleansed"],
        dropna=False
    )
    .agg(
        total_listings=("id", "count"),
        entire_homes=(
            "room_type",
            lambda x: (x == "Entire home/apt").sum()
        ),
        unique_hosts=("host_id", "nunique"),
        median_latitude=("latitude", "median"),
        median_longitude=("longitude", "median")
    )
    .reset_index()
)

neighbourhood_competition["entire_home_share"] = (
    neighbourhood_competition["entire_homes"]
    / neighbourhood_competition["total_listings"]
    * 100
)

display(
    neighbourhood_competition
    .head())


In [ ]:
#Map neighbourhood competition
neighborhood_competition = (
    listings_master
    .groupby(["country", "location", "neighbourhood_cleansed"])
    .agg(
        listing_count=("id", "nunique"),
        latitude=("latitude", "median"),
        longitude=("longitude", "median")
    )
    .reset_index()
)

import plotly.express as px

fig = px.scatter_geo(
    neighborhood_competition,
    lat="latitude",
    lon="longitude",
    size="listing_count",
    color="listing_count",
    hover_name="neighbourhood_cleansed",
    hover_data=["location", "country", "listing_count"],
    scope="europe",
    projection="natural earth",
    color_continuous_scale="YlOrRd",
    title="Airbnb Neighborhood Competition Across Europe"
)

fig.update_layout(
    height=750,
    coloraxis_colorbar_title="Listings"
)

fig.show()

#Save map to drive
map_folder = Path(
    MAP_DIR.as_posix()
)

map_folder.mkdir(parents=True, exist_ok=True)
fig.write_html(
    str(map_folder / "neighborhood_competition_europe.html")
)

print("Interactive map saved successfully!")

In [ ]:
#competition in 1km radius with Barcelona example,
#'competitors_1km' feature for ML
from sklearn.neighbors import BallTree
import numpy as np
import pandas as pd

def add_competition_features(
    df,
    radius_km=1.0,
    group_col="room_type"
):
    result = df.copy()

    result["competitors_1km"] = 0

    earth_radius_km = 6371.0088
    radius_radians = radius_km / earth_radius_km
    for (location, room_type), group in result.groupby(
        ["location", group_col],
        dropna=False
    ):

        if len(group) <= 1:
            continue

        coords = np.radians(
            group[["latitude", "longitude"]].to_numpy()
        )

        tree = BallTree(
            coords,
            metric="haversine"
        )

        counts = tree.query_radius(
            coords,
            r=radius_radians,
            count_only=True
        ) - 1

        result.loc[group.index, "competitors_1km"] = counts

    return result

#Location example
barcelona_data = geo_data[
    geo_data["location"] == "Barcelona"
].copy()

barcelona_data = add_competition_features(
    barcelona_data,
    radius_km=1.0
)

display(
    barcelona_data[
        ["id", "neighbourhood_cleansed",
         "room_type", "competitors_1km"]
    ].head(10)
)

In [ ]:
#Visualization of competition feature
import matplotlib.pyplot as plt

plt.figure(figsize=(11, 8))

scatter = plt.scatter(
    barcelona_data["longitude"],
    barcelona_data["latitude"],
    c=barcelona_data["competitors_1km"],
    cmap="YlOrRd",
    s=5,
    alpha=0.65
)

plt.colorbar(
    scatter,
    label="Comparable Airbnb listings within 1 km"
)

plt.title("Airbnb Competition Density — Barcelona")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.tight_layout()
plt.show()

6. Between Variables  